In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
BASE_PATH = "../input/osic-pulmonary-fibrosis-progression"
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
SAMPLE_SUB = os.path.join(BASE_PATH, "sample_submission.csv")



In [3]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sub_df = pd.read_csv(SAMPLE_SUB)

# rename test columns for clarity
test_df = test_df.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
test_df["Typical_FVC"] = (test_df.Base_FVC.values / test_df.Base_Percent.values) * 100

# ----- compute per‑patient weekly rates (kept for fallback) -----
train_sorted = train_df.sort_values(["Patient", "Weeks"])
train_sorted["FVC_prev"] = train_sorted.groupby("Patient")["FVC"].shift(1)
train_sorted["Weeks_prev"] = train_sorted.groupby("Patient")["Weeks"].shift(1)
train_diff = train_sorted.dropna(subset=["FVC_prev", "Weeks_prev"])
train_diff["weekly_rate"] = (train_diff["FVC"] - train_diff["FVC_prev"]) / (
    train_diff["Weeks"] - train_diff["Weeks_prev"]
)

global_weekly_rate = train_diff["weekly_rate"].mean()
patient_weekly_rate = train_diff.groupby("Patient")["weekly_rate"].mean()

# ----- compute average FVC delta for each week offset from baseline -----
# get each patient’s baseline (earliest week) measurement
baseline = (
    train_df.sort_values(["Patient", "Weeks"])
    .groupby("Patient")
    .first()
    .reset_index()[["Patient", "Weeks", "FVC"]]
    .rename(columns={"Weeks": "Base_Week", "FVC": "Base_FVC"})
)

# merge baseline info back to all training rows
train_merged = train_df.merge(
    baseline, on="Patient", how="left", suffixes=("", "_base")
)
train_merged["week_offset"] = train_merged["Weeks"] - train_merged["Base_Week"]
train_merged["delta"] = train_merged["FVC"] - train_merged["Base_FVC"]

# average delta per offset (can be positive or negative)
offset_mean = train_merged.groupby("week_offset")["delta"].mean()
offset_mean_dict = offset_mean.to_dict()



/tmp/ipykernel_11/262196008.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_diff["weekly_rate"] = (train_diff["FVC"] - train_diff["FVC_prev"]) / (


In [4]:
# split Patient_Week into components
sub_df["Patient"] = sub_df["Patient_Week"].apply(lambda x: x.split("_")[0])
sub_df["Weeks"] = sub_df["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))

# attach baseline info from test set
sub_merged = sub_df.merge(
    test_df[["Patient", "Base_FVC", "Base_Week"]], on="Patient", how="left"
)

# attach patient‑specific rate if available
sub_merged = sub_merged.merge(
    patient_weekly_rate.rename("patient_rate"), on="Patient", how="left"
)

# decide which weekly rate to use (patient‑specific or global)
rate_used = sub_merged["patient_rate"].fillna(global_weekly_rate)

# compute week offset relative to baseline
sub_merged["week_delta"] = sub_merged["Weeks"] - sub_merged["Base_Week"]

# use the offset‑based average delta when available, otherwise fall back to linear rate
sub_merged["offset_delta"] = sub_merged["week_delta"].map(offset_mean_dict)
sub_merged["offset_delta"] = sub_merged["offset_delta"].fillna(
    rate_used * sub_merged["week_delta"]
)

# final FVC prediction
sub_merged["FVC"] = sub_merged["Base_FVC"] + sub_merged["offset_delta"]

# set a higher confidence (200) to better match the Laplace Log Likelihood
sub_merged["Confidence"] = 200



In [5]:
submission = sub_merged[["Patient_Week", "FVC", "Confidence"]].copy()
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
